# SONA - Foto -> The Sims (gratis no Google Colab)

Transforma a foto de um rosto humano real num retrato estilo **The Sims**,
preservando a identidade. Usa a **GPU gratuita** do Colab.

## Passo a passo
1. **Ambiente de execucao -> Alterar o tipo de ambiente** -> **GPU (T4)**.
2. Rode as celulas **na ordem**.
3. 1a vez baixa modelos (~7GB), demora alguns minutos.
4. Ultima celula: envie a foto e veja o resultado.

> Se voce ja rodou antes e deu erro: **Ambiente de execucao -> Reiniciar ambiente** e rode tudo do zero.


## 1) Verificar a GPU


In [ ]:
!nvidia-smi

## 2) Instalar (so o essencial; usa as versoes que ja vem no Colab)
Aviso de 'dependency conflicts' no fim e normal.


In [ ]:
# Conjunto de versoes compativeis entre si (resolve os conflitos do Colab)
!pip -q install huggingface_hub==0.25.2 diffusers==0.27.2 transformers==4.41.2 \
  accelerate==0.34.2 peft==0.13.2 insightface==0.7.3 onnxruntime \
  opencv-python-headless einops
print('OK - bibliotecas instaladas')

## 3) Baixar o InstantID e os modelos


In [ ]:
import os
if not os.path.exists('InstantID'):
    !git clone https://github.com/InstantID/InstantID.git
%cd /content/InstantID
from huggingface_hub import hf_hub_download
hf_hub_download('InstantX/InstantID', 'ControlNetModel/config.json', local_dir='./checkpoints')
hf_hub_download('InstantX/InstantID', 'ControlNetModel/diffusion_pytorch_model.safetensors', local_dir='./checkpoints')
hf_hub_download('InstantX/InstantID', 'ip-adapter.bin', local_dir='./checkpoints')
print('OK - modelos InstantID baixados')

## 4) Carregar o detector de rosto e o gerador (1a vez: alguns minutos)


In [ ]:
import cv2, torch, numpy as np, os, glob, shutil
from PIL import Image
from insightface.app import FaceAnalysis
from diffusers.models import ControlNetModel
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps


# Baixa e extrai o antelopev2 na estrutura correta (evita o bug do insightface)
import zipfile, urllib.request
if not os.path.exists('models/antelopev2/scrfd_10g_bnkps.onnx'):
    os.makedirs('models', exist_ok=True)
    _z='models/antelopev2.zip'
    if not os.path.exists(_z):
        urllib.request.urlretrieve('https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip', _z)
    with zipfile.ZipFile(_z) as _zf: _zf.extractall('models')

app = FaceAnalysis(name='antelopev2', root='./', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

controlnet = ControlNetModel.from_pretrained('./checkpoints/ControlNetModel', torch_dtype=torch.float16)
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    'wangqixun/YamerMIX_v8', controlnet=controlnet, torch_dtype=torch.float16)
pipe.load_ip_adapter_instantid('./checkpoints/ip-adapter.bin')
pipe.enable_model_cpu_offload()
print('OK - pronto para gerar')

## 5) Estilo The Sims (prompt)


In [ ]:
PROMPT = ('official character portrait in the art style of The Sims 3 video game, '
          'stylized 3D rendered character, smooth CG skin shading, soft studio lighting, '
          'clean simple background, Maxis art style, high quality 3D render, head and shoulders portrait')
NEGATIVE = ('photorealistic, real photo, photograph, anime, 2d, cartoon sketch, deformed, '
            'distorted, disfigured, extra limbs, lowres, blurry, bad anatomy, watermark, text, '
            'different person, multiple faces')

## 6) Enviar a foto e gerar


In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
up = files.upload()
fname = list(up.keys())[0]
face_image = Image.open(fname).convert('RGB')

info = app.get(cv2.cvtColor(np.array(face_image), cv2.COLOR_RGB2BGR))
if not info:
    raise SystemExit('REJEITADA: nenhum rosto humano detectado. Envie outra foto.')
info = sorted(info, key=lambda x:(x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
face_emb = info['embedding']
face_kps = draw_kps(face_image, info['kps'])

pipe.set_ip_adapter_scale(0.8)
result = pipe(prompt=PROMPT, negative_prompt=NEGATIVE,
    image_embeds=face_emb, image=face_kps,
    controlnet_conditioning_scale=0.8, ip_adapter_scale=0.8,
    num_inference_steps=30, guidance_scale=5).images[0]

fig, ax = plt.subplots(1, 2, figsize=(11,6))
ax[0].imshow(face_image); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(result); ax[1].set_title('The Sims'); ax[1].axis('off')
plt.show()
result.save('sona_resultado.png'); print('Salvo como sona_resultado.png')